In [11]:
import os
import json
from tavily import TavilyClient
from openai import OpenAI
from typing import List, Dict
import csv
from trulens.core import TruSession
from trulens.apps.custom import instrument
from dotenv import load_dotenv

load_dotenv()


True

In [12]:
session = TruSession()

## Uncomment the following to reset database 
session.reset_database()

Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


In [13]:
# with open("../../GroundTruths_Dataset -No Multihop No yes or no questions.csv", mode='r', encoding='utf-8') as file:
#     csv_reader = csv.DictReader(file)
#     # Iterate through rows as dictionaries
#     queries = []
#     for row in csv_reader:
#         queries.append(row["query"]) 

with open("../../GroundTruths_Dataset - Sheet1.csv", mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)
    # Iterate through rows as dictionaries
    queries = []
    for row in csv_reader:
        queries.append(row["query"]) 

len(queries)

23

In [14]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILYAI_API_KEY"))
llm = OpenAI()

In [15]:
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("openai-test")

In [16]:
embed = llm.embeddings.create


In [31]:
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query):
            k=3
            embedding=self.embed( model="text-embedding-3-large",input=query).data[0].embedding

            vecs = self.index.query(
            vector=embedding,
            top_k=k,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            new_docs = self._generate_docs(query)
            docs.extend(new_docs)
            return docs

                 
        
        def _generate_docs(self, query, m=2):
             res =  llm.chat.completions.create(
  model="gpt-4o",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "You are a highly reliable assistant tasked with answering questions by generating m clear, concise, and accurate passages. Your responses must adhere to the following rules:\n\nGenerate up to m passages that directly and thoroughly answer the given question. Each passage should provide unique, complementary information to enhance understanding.\nIf the question cannot be confidently answered based on your training data or knowledge, you must strictly respond with \"I don't know\". Do not attempt to guess or fabricate information.\nIf the question is unclear or ambiguous, provide a response that explains the issue and ask for clarification.\nDo nto provide answers you are not confident in. For example do not use May or maybe \nExample Format:\n\nInput: [Insert the question]\nOutput:\n [Answer content]\n [Answer content]\n...\nIf no confident answer exists: \"I don't know\""
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": f"entityy: MOHAP (Ministry of Health and Public Prevention UAE), Query {query}, Number of documents {m}"
        }
      ]
    }
  ],
  response_format={
    "type": "json_schema",
    "json_schema": {
      "name": "string_list",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": {
          "strings": {
            "type": "array",
            "description": "A list of strings.",
            "items": {
              "type": "string"
            }
          }
        },
        "required": [
          "strings"
        ],
        "additionalProperties": False
      }
    }
  },
  temperature=0,
  max_completion_tokens=10000,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0
)
             parsed_res = json.loads(res.choices[0].message.content)
             print(len(parsed_res))
             print(parsed_res)

             return parsed_res["strings"]


In [32]:
openai_retreiver = retriever(embed, index)

In [33]:
## Test
query="what is the the service i use to register as new practicing doctor in the UAE "
res = openai_retreiver.get_data(query)
print(res)
print(type(res))

1
{'strings': ["To register as a new practicing doctor in the UAE through MOHAP, you need to use the 'Issuing a Medical License' service. This service is designed for healthcare professionals seeking to obtain a license to practice medicine in the UAE.", 'The process typically requires the submission of several documents, including proof of qualifications and professional experience. However, the exact number and type of documents may vary, so it is advisable to consult the MOHAP website or contact their customer service for the most accurate and up-to-date information.']}
['doctor to practice in the medical or dental profession for a limited period of time in a private health facility.Service Process1Login to the MOHAP website or smart app using the UAE PASS to apply for the service.2The customer (facility) must login through the account of the licensed facility and provide all the required information and documents as per the type of license.3The customer must refer the application t

In [34]:
len(res)

5

In [35]:
res

['doctor to practice in the medical or dental profession for a limited period of time in a private health facility.Service Process1Login to the MOHAP website or smart app using the UAE PASS to apply for the service.2The customer (facility) must login through the account of the licensed facility and provide all the required information and documents as per the type of license.3The customer must refer the application to the Ministry of Health and Prevention.4The employee concerned will check the application. If the',
 'doctor from the UAE to practice in the medical or dental profession for a limited period in a private health facility.Service Process1Login to the MOHAP website or smart app using the UAE PASS to apply for the service.2The customer (facility) must login through the account of the licensed facility and provide all the required information and documents as per the type of license.3The customer must refer the application to the Ministry of Health and Prevention.4The employee 

In [36]:
prompt ="Given the provided context, generate a response that is accurate, concise, and strictly aligned with the information retrieved. Ensure the response does not include hallucinations, speculations, or unsupported claims. The response should be neutral, fact-based, and respectful, especially when addressing sensitive or ambiguous topics. If the context provided is insufficient, clearly state that more information is needed. Prioritize safety and relevance, and avoid generating offensive or harmful content. Please the answer should be in paragraph style and sentences. do not introduce lists and bullet points"

class generator:
    def __init__(self, llm):
        self.llm = llm
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": query+formatted_context
        }
    ]
)
        return response.choices[0].message

In [38]:
llm = OpenAI(api_key=os.getenv("OPEN_AI_EVAL_KEY"))
gen = generator(llm)


In [39]:
gen.generate(queries[15], openai_retreiver.get_data(queries[15]))

1
{'strings': ["A copy of the medical staff's valid passport.", "A letter from the current employer confirming the staff's employment and good standing status."]}


ChatCompletionMessage(content="The requirements for obtaining a good standing certificate for medical staff, which is fee-exempt for renewing staff licenses, include several key documents. These documents are: a letter from the facility requesting the re-licensing of the doctor, the doctor's contract of employment, a copy of the doctor's valid license, an assessment certificate, the facility's plan, a certificate of good conduct for the doctor, and a medical fitness certificate if the doctor is 60 years or older. Additionally, a copy of the medical staff's valid passport and a letter from their current employer confirming their employment and good standing status are required. The certificate is not issued to trainees, visitors, or those with only an initial license, and the applicant must have been licensed by the Ministry of Health and Prevention for more than three months without any prohibitive medical offenses. The good standing certificate is valid for six months from the date of

In [41]:
class Rag_app:
    def __init__(self,llm,retriever):
        self.retriever = retriever
        self.llm=llm
    @instrument
    def retrieve(self, query: str) -> List[str]:
        """
        Method to handle document retrieval.
        IMPORTANT: The method name 'retrieve' will be used in selectors
        """
        documents = self.retriever.get_data(query)
        return documents
    
    @instrument
    def generate(self, query: str, context: List[str]) -> str:
        """
        Method to handle response generation.
        IMPORTANT: The method name 'generate' will be used in selectors
        """
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate(query ,formatted_context)
        return response
    
    
    @instrument
    def query(self, question: str) -> Dict:
        """
        Main method that orchestrates the RAG pipeline.
        IMPORTANT: Return keys must match selector paths
        """
        context = self.retrieve(question)
        response = self.generate(question, context)
        
        return response.content

In [42]:
rag_app = Rag_app(generator(llm), openai_retreiver)
# provider = OpenAI(model_engine="gpt-4o", api_key=os.getenv("OPEN_AI_EVAL_KEY"))

import numpy as np
from trulens.core import Feedback
from trulens.core import Select
from trulens.providers.openai import OpenAI
provider = OpenAI()

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [49]:
from trulens.apps.custom import TruCustomApp
##NAMING CONVENTION eval-{Retriever}-{generator}-{chunksize}-@{k}

tru_rag = TruCustomApp(
    rag_app,
    app_name="Astute RAG",
    app_version="4o-large_3-500-Multihop&Yes-No",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [ ]:
# rag_app.query(queries[19])

In [44]:
from trulens.core.utils.pace import Pace

# Define your desired pacing rate
pace = Pace(marks_per_second=0.5, seconds_per_period=30.0)

In [ ]:
queries[15:]

In [52]:
with tru_rag as recording:
    for eval in queries[15:]:
        print(eval)
        pace.mark()
        rag_app.query(
        eval
    )

What are the requirement documents for the good standing certificate of medical staff in the sector the is fee-exempt for renewal staff licenses?
1
{'strings': ['A copy of the valid medical license issued by the Ministry of Health and Prevention (MOHAP).', "A letter from the current employer confirming the medical staff's employment status and stating that the renewal is fee-exempt."]}
 I have a medical equipment that is manufactured from animal-based products, What is the condition to renew the license? 
1
{'strings': ['To renew the license for medical equipment manufactured from animal-based products in the UAE, you must comply with specific regulatory requirements set by MOHAP. These typically include providing documentation that verifies the safety and efficacy of the product, as well as evidence of compliance with any relevant international standards or guidelines. Additionally, you may need to submit a declaration regarding the source and processing of the animal-based materials 

In [47]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

Starting dashboard ...


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://192.168.1.12:2503 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>